In [ ]:
!pip install streamlit requests pyngrok --quiet

!wget -qO - https://ngrok-agent.s3.amazonaws.com/ngrok.asc | sudo tee /etc/apt/trusted.gpg.d/ngrok.asc >/dev/null \
  && echo "deb https://ngrok-agent.s3.amazonaws.com/ buster main" | sudo tee /etc/apt/sources.list.d/ngrok.list \
  && sudo apt-get update \
  && sudo apt-get install ngrok -y --quiet

In [ ]:
from getpass import getpass
import os

NGROK_TOKEN = getpass("Ingresa tu token de ngrok: ")

os.system(f"ngrok config add-authtoken {NGROK_TOKEN}")

In [ ]:
%%writefile app.py
import streamlit as st
import requests
import json
from datetime import datetime

# ============================================================
# CONFIGURACIÓN GENERAL
# ============================================================
st.set_page_config(
    page_title="Goles y Troncos | Reservas",
    page_icon="⚽",
    layout="wide"
)

# ============================================================
# ESTILOS CSS PERSONALIZADOS
# ============================================================
st.markdown("""
<style>
    .stApp {
        background: linear-gradient(135deg, #061b0b 0%, #0b3d18 45%, #111111 100%);
        color: white;
    }

    .main-title {
        font-size: 52px;
        font-weight: 900;
        color: #ffffff;
        text-align: center;
        margin-bottom: 0px;
        text-shadow: 2px 2px 8px #000000;
    }

    .subtitle {
        font-size: 20px;
        text-align: center;
        color: #d8ffd8;
        margin-bottom: 30px;
    }

    .hero-box {
        background: rgba(255, 255, 255, 0.08);
        border: 1px solid rgba(255, 255, 255, 0.18);
        border-radius: 22px;
        padding: 28px;
        margin-bottom: 25px;
        box-shadow: 0 8px 32px rgba(0,0,0,0.35);
    }

    .card {
        background: rgba(0, 0, 0, 0.35);
        border-radius: 18px;
        padding: 22px;
        border: 1px solid rgba(255,255,255,0.12);
        min-height: 145px;
    }

    .card h3 {
        color: #9DFF8A;
        margin-top: 0px;
    }

    .chat-container {
        background: rgba(255,255,255,0.09);
        border-radius: 20px;
        padding: 20px;
        border: 1px solid rgba(255,255,255,0.15);
    }

    .user-bubble {
        background: #1f7a3a;
        color: white;
        padding: 14px 18px;
        border-radius: 18px 18px 4px 18px;
        margin: 10px 0px;
        max-width: 80%;
        margin-left: auto;
        box-shadow: 0px 4px 12px rgba(0,0,0,0.25);
    }

    .bot-bubble {
        background: #f4f4f4;
        color: #111111;
        padding: 14px 18px;
        border-radius: 18px 18px 18px 4px;
        margin: 10px 0px;
        max-width: 80%;
        margin-right: auto;
        box-shadow: 0px 4px 12px rgba(0,0,0,0.25);
    }

    .small-text {
        color: #d0d0d0;
        font-size: 14px;
        text-align: center;
    }

    .footer {
        color: #d8ffd8;
        text-align: center;
        margin-top: 35px;
        font-size: 14px;
    }

    div[data-testid="stSidebar"] {
        background-color: #071b0c;
    }
</style>
""", unsafe_allow_html=True)

# ============================================================
# ESTADO DE SESIÓN
# ============================================================
if "messages" not in st.session_state:
    st.session_state.messages = [
        {
            "role": "assistant",
            "content": "¡Hola! Bienvenido a Goles y Troncos ⚽. Puedo ayudarte a reservar, cancelar o consultar información sobre nuestras canchas sintéticas."
        }
    ]

if "technical_history" not in st.session_state:
    st.session_state.technical_history = []

# ============================================================
# BARRA LATERAL
# ============================================================
st.sidebar.title("⚙️ Configuración")

WEBHOOK_URL = st.sidebar.text_input(
    "URL del Webhook de n8n",
    value=""
)

SESSION_ID = st.sidebar.text_input(
    "Session ID",
    value="cliente-web-001"
)

st.sidebar.markdown("---")
st.sidebar.subheader("📌 Pruebas rápidas")

quick_examples = [
    "Hola, soy Juan Pérez, mi celular es 1234567890 y quiero reservar fútbol 8 mañana a las 10 pm.",
    "Quiero reservar una cancha de fútbol 8 mañana a las 10 pm.",
    "¿Qué incluye la cancha de fútbol 8?",
    "¿Hay parqueadero?",
    "¿Qué pasa si pierdo un balón?",
    "Hola, quiero cancelar la cancha de fútbol 8 mañana a las 10 pm, a nombre de Juan Pérez, con teléfono 1234567890."
]

selected_example = st.sidebar.selectbox(
    "Selecciona un ejemplo",
    quick_examples
)

if st.sidebar.button("Usar ejemplo"):
    st.session_state.example_to_use = selected_example

st.sidebar.markdown("---")
st.sidebar.info(
    "Proyecto final de Procesamiento de Lenguaje Natural. "
    "Frontend en Streamlit, backend en n8n, IA con Gemini y datos en Google Sheets/Drive."
)

# ============================================================
# ENCABEZADO
# ============================================================
st.markdown('<div class="main-title">⚽ Goles y Troncos</div>', unsafe_allow_html=True)
st.markdown(
    '<div class="subtitle">Reservas inteligentes de canchas sintéticas en Manizales</div>',
    unsafe_allow_html=True
)

st.markdown("""
<div class="hero-box">
    <h2 style="text-align:center; color:#9DFF8A;">Reserva, consulta o cancela desde nuestro chat inteligente</h2>
    <p style="text-align:center; font-size:17px;">
        Canchas de fútbol 5 y fútbol 8 disponibles para parches, torneos, amigos y empresas.
    </p>
    <p class="small-text">
        Propietarios: Juan Pablo Ocampo y Juan Esteban Mora · Manizales, Caldas
    </p>
</div>
""", unsafe_allow_html=True)

# ============================================================
# TARJETAS INFORMATIVAS
# ============================================================
col1, col2, col3 = st.columns(3)

with col1:
    st.markdown("""
    <div class="card">
        <h3>🥅 Fútbol 5</h3>
        <p>Ideal para partidos rápidos, grupos pequeños y encuentros entre amigos.</p>
        <strong>Valor:</strong> $90.000 COP / hora
    </div>
    """, unsafe_allow_html=True)

with col2:
    st.markdown("""
    <div class="card">
        <h3>🏟️ Fútbol 8</h3>
        <p>Perfecta para partidos más amplios, entrenamientos y torneos recreativos.</p>
        <strong>Valor:</strong> $140.000 COP / hora
    </div>
    """, unsafe_allow_html=True)

with col3:
    st.markdown("""
    <div class="card">
        <h3>🤖 Chatbot IA</h3>
        <p>Consulta disponibilidad, agenda reservas, cancela y pregunta por implementos incluidos.</p>
        <strong>Canal:</strong> Web
    </div>
    """, unsafe_allow_html=True)

st.markdown("---")

# ============================================================
# FUNCIÓN PARA ENVIAR MENSAJE A N8N
# ============================================================
def send_to_n8n(message: str):
    payload = {
        "sessionId": SESSION_ID,
        "chatInput": message,
        "channel": "web"
    }

    try:
        response = requests.post(
            WEBHOOK_URL,
            json=payload,
            headers={"Content-Type": "application/json"},
            timeout=60
        )

        if response.status_code != 200:
            return {
                "reply": f"Error al conectar con n8n. Código de estado: {response.status_code}",
                "raw": response.text,
                "ok": False
            }

        data = response.json()

        if isinstance(data, list) and len(data) > 0:
            data = data[0]

        reply = data.get("reply", "No recibí una respuesta válida del sistema.")

        return {
            "reply": reply,
            "raw": data,
            "ok": True
        }

    except Exception as e:
        return {
            "reply": f"No pude conectar con el servicio de reservas. Detalle: {str(e)}",
            "raw": {},
            "ok": False
        }

# ============================================================
# CHATBOX PRINCIPAL
# ============================================================
left, right = st.columns([2, 1])

with left:
    st.markdown('<div class="chat-container">', unsafe_allow_html=True)
    st.subheader("💬 Chat de reservas")

    for msg in st.session_state.messages:
        if msg["role"] == "user":
            st.markdown(
                f'<div class="user-bubble">{msg["content"]}</div>',
                unsafe_allow_html=True
            )
        else:
            st.markdown(
                f'<div class="bot-bubble">{msg["content"]}</div>',
                unsafe_allow_html=True
            )

    default_text = st.session_state.pop("example_to_use", "") if "example_to_use" in st.session_state else ""

    user_input = st.chat_input("Escribe tu mensaje aquí... Ej: Quiero reservar fútbol 8 mañana a las 10 pm")

    if default_text:
        user_input = default_text

    if user_input:
        st.session_state.messages.append({
            "role": "user",
            "content": user_input
        })

        with st.spinner("El asistente está revisando tu solicitud..."):
            result = send_to_n8n(user_input)

        st.session_state.messages.append({
            "role": "assistant",
            "content": result["reply"]
        })

        raw = result.get("raw", {})
        st.session_state.technical_history.append({
            "hora": datetime.now().strftime("%H:%M:%S"),
            "mensaje": user_input,
            "reply": result["reply"],
            "intent": raw.get("intent"),
            "sheet_updated": raw.get("sheet_updated"),
            "available": raw.get("available"),
            "status": raw.get("status"),
            "reservation_id": raw.get("reservation_id")
        })

        st.rerun()

    st.markdown('</div>', unsafe_allow_html=True)

with right:
    st.subheader("📊 Estado técnico")

    if st.session_state.technical_history:
        last = st.session_state.technical_history[-1]

        st.metric("Intención", last.get("intent") or "N/A")
        st.metric("Estado", last.get("status") or "N/A")
        st.metric("Actualizó Sheet", str(last.get("sheet_updated")))
        st.metric("Disponible", str(last.get("available")))

        if last.get("reservation_id"):
            st.success(f"ID reserva: {last.get('reservation_id')}")
    else:
        st.info("Aún no hay interacciones registradas.")

    with st.expander("🧪 Historial técnico JSON"):
        st.json(st.session_state.technical_history)

# ============================================================
# SECCIÓN DE AYUDA
# ============================================================
st.markdown("---")
st.subheader("🧭 ¿Qué puedes preguntarle al chatbot?")

c1, c2, c3 = st.columns(3)

with c1:
    st.markdown("""
    **Reservar**
    - Quiero reservar fútbol 8 mañana a las 10 pm.
    - Soy Juan Pérez, mi celular es 3001234567 y quiero reservar fútbol 5 hoy a las 7 pm.
    """)

with c2:
    st.markdown("""
    **Cancelar**
    - Quiero cancelar mi reserva.
    - Cancela la cancha de fútbol 8 mañana a las 10 pm a nombre de Juan Pérez.
    """)

with c3:
    st.markdown("""
    **Preguntas frecuentes**
    - ¿Qué incluye la cancha de fútbol 8?
    - ¿Hay parqueadero?
    - ¿Qué pasa si pierdo un balón?
    """)

st.markdown("""
<div class="footer">
    ⚽ Goles y Troncos · Sistema académico de reservas inteligentes · Manizales, Colombia
</div>
""", unsafe_allow_html=True)

In [ ]:
import os
import time
from pyngrok import ngrok

# Cerrar túneles anteriores
ngrok.kill()

# Levantar Streamlit en segundo plano
os.system("streamlit run app.py --server.port 8501 &")

time.sleep(5)

# Crear túnel público
public_url = ngrok.connect(8501, proto="http")

print("===================================================")
print("🚀 PÁGINA WEB DESPLEGADA")
print("===================================================")
print("Abre este enlace:")
print(public_url.public_url)
print("===================================================")